# 🚚 Olist Delivery Delay Prediction - End to End ML Project
**Problem:** Order place hote hi predict karo ki wo estimated date tak deliver hoga ya **late** hoga, aur delivery kitne din me hogi.

**Models:** Logistic Regression, Decision Tree, Random Forest (classification) | Linear Regression, Decision Tree, Random Forest (regression) | KMeans (segmentation)

**Pipeline:** Load → Merge → EDA → Preprocessing → Models (Train vs Test) → Evaluation → Business Impact → Conclusion
## 1. Libraries Import

In [6]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
import joblib 
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,r2_score,mean_absolute_error,confusion_matrix,roc_curve)

## 2. Data Load

In [7]:
orders = pd.read_csv("olist_orders_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
products = pd.read_csv("olist_products_Dataset.csv")
translation = pd.read_csv("Product_category_name_translation.csv")
geo = pd.read_csv("olist_geolocation_dataset.csv")

In [8]:
# CSV files same folder me hain to path khali rehne do

orders = pd.read_csv("olist_orders_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")
geo = pd.read_csv("olist_geolocation_dataset.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [17]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [9]:
orders.shape

(99441, 8)

## 3. Cleaning - Orders
Sirf **delivered** orders lo, aur 2 targets banao:
- `is_late` → 1 agar estimated date ke baad aaya, warna 0 (classification)
- `delivery_days` → kitne din lage (regression)

In [16]:
df = orders[orders["order_status"] == "delivered"].copy()
df = df.dropna(subset=["order_delivered_customer_date"])

# date wale columns ko date type me badlo
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df[col] = pd.to_datetime(df[col])

df["is_late"] = (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(int)
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days

df[["order_id", "is_late", "delivery_days"]].head()

,order_id,is_late,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,0,8
1,53cdb2fc8bc7dce0b6741e2150273451,0,13
2,47770eb9100c2d0c44946d9cf07ec65d,0,9
3,949d5b44dbf5de918fe9c16f97b45f8a,0,13
4,ad21c59c0840e6cb83a9ceb5573f8159,0,2


## 4. Cleaning - Items, Products, Geolocation
Ek order me kai items ho sakte hain, isliye `order_id` ke hisaab se group karte hain.

In [ ]:
# Items: har order ki ek row 
items_grouped = items.groupby('order_id').agg({'price':'sum','freight_value':'sum','order_item_id':'count','seller_id':'first','product_id':'first'}).reset_index()
itmes_grouped = items_grouped.rename(columns={'freight_value': 'freight','order_item_id':'total_items'})

products = products.merge(translation,on='product_category_name',how='left')
products['volume_cm3']

In [ ]:
# Items: har order ki ek row
items_grouped = items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "order_item_id": "count",
    "seller_id": "first",
    "product_id": "first"
}).reset_index()
items_grouped = items_grouped.rename(columns={"freight_value": "freight", "order_item_id": "total_items"})

# Products: english category + volume
products = products.merge(translation, on="product_category_name", how="left")
products["volume_cm3"] = products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"]
products = products.rename(columns={"product_category_name_english": "category"})
products = products[["product_id", "category", "product_weight_g", "volume_cm3"]]

# Geo: ek zip code ke kai lat/lng hain, to average le lo
geo_avg = geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean().reset_index()
geo_avg.head()